# Entraînement du classificateur de titres

Ce notebook entraîne et compare deux pipelines sur les titres AG News. L’application Streamlit utilise la régression logistique, qui a obtenu le meilleur résultat sur `test.csv` lors de l’audit. Lancez Jupyter depuis le dossier du projet pour que les chemins vers les CSV soient corrects.


In [ ]:
from pathlib import Path
import re
import joblib
import pandas as pd
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

BASE_DIR = Path.cwd()
train_df = pd.read_csv(BASE_DIR / "train.csv")
test_df = pd.read_csv(BASE_DIR / "test.csv")

class_names = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
print("Train:", train_df.shape, " Test:", test_df.shape)
print("Class counts:", train_df["Class Index"].value_counts().sort_index().to_dict())


## Prétraitement

Le code de l’application applique exactement la même règle : minuscules, conservation des lettres anglaises, lemmatisation spaCy et retrait des mots vides anglais. `nlp.pipe` traite les titres par lots, ce qui est plus rapide qu’un appel spaCy séparé pour chaque ligne.


In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess_many(texts):
    cleaned = [re.sub(r"[^a-z\s]", "", str(text).casefold()) for text in texts]
    docs = nlp.pipe(cleaned, batch_size=512)
    return [
        " ".join(token.lemma_ for token in doc if token.text not in STOP_WORDS)
        for doc in docs
    ]

train_df["clean_title"] = preprocess_many(train_df["Title"].tolist())
test_df["clean_title"] = preprocess_many(test_df["Title"].tolist())
print(train_df[["Title", "clean_title", "Class Index"]].head())


In [ ]:
X_train = train_df["clean_title"]
y_train = train_df["Class Index"]
X_test = test_df["clean_title"]
y_test = test_df["Class Index"]


## Modèle de référence : Naive Bayes

Le pipeline associe un TF-IDF limité à 5 000 caractéristiques (mots seuls et paires de mots) à `MultinomialNB`. Le résultat est conservé comme référence dans `news_classifier.pkl`; l’application ne le charge pas.


In [ ]:
nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("nb", MultinomialNB()),
])
nb_pipeline.fit(X_train, y_train)
nb_predictions = nb_pipeline.predict(X_test)
print("Naive Bayes accuracy:", accuracy_score(y_test, nb_predictions))
print(classification_report(
    y_test,
    nb_predictions,
    labels=[1, 2, 3, 4],
    target_names=[class_names[i] for i in [1, 2, 3, 4]],
))
joblib.dump(nb_pipeline, BASE_DIR / "news_classifier.pkl")


## Modèle utilisé par l’application : Logistic Regression

La régression logistique utilise le même TF-IDF et `max_iter=1000`. Le pipeline complet est sauvegardé dans `news_classifier_lr.pkl`, avec son vectoriseur intégré.


In [ ]:
lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("lr", LogisticRegression(max_iter=1000, random_state=42)),
])
lr_pipeline.fit(X_train, y_train)
lr_predictions = lr_pipeline.predict(X_test)
print("Logistic Regression accuracy:", accuracy_score(y_test, lr_predictions))
print(classification_report(
    y_test,
    lr_predictions,
    labels=[1, 2, 3, 4],
    target_names=[class_names[i] for i in [1, 2, 3, 4]],
))
joblib.dump(lr_pipeline, BASE_DIR / "news_classifier_lr.pkl")


## Exemple de prédiction

La dernière cellule reproduit l’inférence de l’application : le titre est prétraité, puis le pipeline renvoie une classe et des scores par classe.


In [ ]:
headline = "NASA spacecraft discovers signs of water beneath the surface of Mars"
clean_headline = preprocess_many([headline])[0]
probabilities = lr_pipeline.predict_proba([clean_headline])[0]
predicted_id = int(lr_pipeline.classes_[probabilities.argmax()])
print("Titre :", headline)
print("Catégorie :", class_names[predicted_id])
print({class_names[int(label)]: round(float(score), 4)
       for label, score in zip(lr_pipeline.classes_, probabilities)})


## À noter

Les scores produits par `predict_proba` sont des estimations du modèle, pas des probabilités calibrées. Le jeu de test fourni contient 7 600 titres; son exactitude mesurée lors de l’audit pour la régression logistique est d’environ 85,1 %. Cette valeur concerne ces données et ne garantit pas le même résultat sur d’autres sources ou sujets.
